# ⚡ HypeCut — Mobile Live & VOD Clipper (Free on Google Colab)

Clip the moments where stream chat explodes into **30s vertical 9:16 reels** with word-by-word captions, hooks, and viral scores — right from your phone, with **no laptop needed**.

**How to use on Mobile:**
1. Tap **Runtime → Change runtime type → T4 GPU** (optional, makes Whisper 5× faster)
2. Run **Step 1** once to install HypeCut & FFmpeg
3. In **Step 2**, upload an `.mp4` from your phone (or paste a stream/VOD link) and tap **Play** ▶
4. Run **Step 3** to download all your clips as a ZIP file to your phone

> ⚠️ **Clip Responsibly:** Only clip your own streams or creators who explicitly allow clipping.

In [ ]:
#@title 1. Install HypeCut & FFmpeg (Run Once) { display-mode: "form" }
import os, subprocess, sys

print("⏳ Installing ffmpeg & cloning HypeCut...")
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=True)

if not os.path.exists("/content/hypecut"):
    subprocess.run(["git", "clone", "https://github.com/kunalmhetre27-ui/hypecut.git", "/content/hypecut"], check=True)
else:
    subprocess.run(["git", "-C", "/content/hypecut", "pull", "--ff-only"], check=False)

# Install CLI dependencies (skip pywebview desktop GUI which is not needed in Colab)
pkgs = [
    "numpy>=1.24",
    "faster-whisper>=1.0",
    "yt-dlp>=2024.1.1",
    "opencv-python-headless>=4.8,<5",
    "chat-downloader>=0.2.8",
    "streamlink>=6.0",
    "google-genai"
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "/content/hypecut"], check=True)
print("✅ HypeCut installed and ready! Now run Step 2.")

In [ ]:
#@title 2. Upload Video or Paste Link & Run Clipper { display-mode: "form" }
input_source = "Upload Video from Phone (.mp4)" #@param ["Upload Video from Phone (.mp4)", "Stream / VOD Link (YouTube or Twitch)", "Existing Colab File Path"]
stream_url_or_path = "" #@param {type:"string"}
mode = "vod" #@param ["vod", "live"]
run_demo_instead = False #@param {type:"boolean"}
max_clips = 6 #@param {type:"slider", min:1, max:10, step:1}
max_clip_seconds = 30 #@param {type:"slider", min:15, max:45, step:5}
language = "auto" #@param ["auto", "en", "hi"]
hinglish_captions = False #@param {type:"boolean"}
layout = "blur" #@param ["blur", "crop"]
caption_style = "bold" #@param ["bold", "clean", "box"]
whisper_model = "base" #@param ["tiny", "base", "small", "medium"]
gemini_api_key = "" #@param {type:"string"}

import os, shutil, subprocess
from google.colab import files
from IPython.display import display, Markdown

OUT_DIR = "/content/hypecut_clips"
if os.path.exists(OUT_DIR):
    shutil.rmtree(OUT_DIR)
os.makedirs(OUT_DIR, exist_ok=True)

env = os.environ.copy()
if gemini_api_key.strip():
    env["GEMINI_API_KEY"] = gemini_api_key.strip()

if run_demo_instead:
    cmd = ["hypecut", "--demo", "--out", OUT_DIR]
    if not gemini_api_key.strip():
        cmd.append("--no-gemini")
else:
    target_input = ""
    if input_source == "Upload Video from Phone (.mp4)":
        print("📤 Tap 'Choose Files' below to upload your .mp4 video from your phone...")
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("❌ No file uploaded. Run Step 2 again and select an .mp4 file.")
        uploaded_name = list(uploaded.keys())[0]
        target_input = os.path.join("/content", uploaded_name)
        print(f"✅ Uploaded: {target_input}")
    elif input_source == "Existing Colab File Path":
        clean_path = stream_url_or_path.strip().strip('"').strip("'")
        if not clean_path or not os.path.exists(clean_path):
            raise FileNotFoundError(f"❌ File not found at: '{clean_path}'. Check the path in the left Files panel.")
        target_input = clean_path
    else:
        if not stream_url_or_path.strip():
            raise ValueError("❌ Please paste a YouTube or Twitch URL in stream_url_or_path (or switch input_source to 'Upload Video from Phone (.mp4)').")
        target_input = stream_url_or_path.strip()

    cmd = [
        "hypecut", target_input,
        "--out", OUT_DIR,
        "--max-clips", str(max_clips),
        "--max-clip", str(max_clip_seconds),
        "--layout", layout,
        "--caption-style", caption_style,
        "--whisper-model", whisper_model,
    ]
    if mode == "live" and input_source == "Stream / VOD Link (YouTube or Twitch)":
        cmd.extend(["--mode", "live"])
    if language != "auto":
        cmd.extend(["--language", language])
    if hinglish_captions:
        cmd.append("--hinglish")
    if not gemini_api_key.strip():
        cmd.append("--no-gemini")

print("🚀 Running:", " ".join(cmd))
proc = subprocess.Popen(cmd, cwd="/content/hypecut", env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
ret = proc.wait()
if ret != 0:
    raise RuntimeError(f"HypeCut exited with code {ret}. Check the log above.")

mp4s = [f for f in os.listdir(OUT_DIR) if f.endswith(".mp4")] if os.path.exists(OUT_DIR) else []
if not mp4s:
    raise RuntimeError(
        "❌ 0 clips generated!\n"
        "👉 If you used a YouTube link, YouTube likely blocked Colab's IP. Download the video to your phone, set input_source to 'Upload Video from Phone (.mp4)', and run Step 2 again.\n"
        "👉 If you uploaded a video, make sure it has clear spoken audio and is longer than max_clip_seconds."
    )

index_path = os.path.join(OUT_DIR, "index.md")
if os.path.exists(index_path):
    with open(index_path, "r", encoding="utf-8") as f:
        display(Markdown(f.read()))
print(f"✅ Done! {len(mp4s)} clip(s) ready. Run Step 3 to download your clips.")

In [ ]:
#@title 3. Download All Clips (.zip) to Phone { display-mode: "form" }
import os, shutil
from google.colab import files

OUT_DIR = "/content/hypecut_clips"
ZIP_PATH = "/content/hypecut_clips.zip"

mp4s = [f for f in os.listdir(OUT_DIR) if f.endswith(".mp4")] if os.path.exists(OUT_DIR) else []
if not mp4s:
    print("❌ No clips found yet. Run Step 2 first!")
else:
    shutil.make_archive("/content/hypecut_clips", "zip", OUT_DIR)
    print(f"📦 Packed {len(mp4s)} clip(s) + captions into {ZIP_PATH}")
    files.download(ZIP_PATH)